# ST554 Final Project: Siona Benjamin
For the final project we'll use spark to handle streaming data and fitting a machine learning model. The data used below describes power consumption from different time zones of Tetoauan city in relation to factors such as time of day, temperature, and humidity. 

To get started, we'll read in our data as a pandas dataframe, and then convert this to a spark dataframe.

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder, CrossValidatorModel
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.regression import LinearRegression
from pyspark.sql.types import StructType
from pyspark.sql.functions import col
from pyspark.ml.feature import SQLTransformer, PCA, Binarizer, OneHotEncoder, VectorAssembler, StringIndexer

In [2]:
#create spark session
spark = SparkSession.builder.appName("final_project").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 12:04:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/29 12:04:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [70]:
#import data as pandas dataframe
power_data = pd.read_csv('power_ml_data.csv')
#convert pandas dataframe to spark dataframe
power_df = spark.createDataFrame(power_data)

Using `.show()` we can see what our data columns look like while `.dtypes` lets us see what data type each column is stored as.

In [4]:
power_df.show(10)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
|      5.853|    76.9|     0.081|       

In [5]:
power_df.dtypes

[('Temperature', 'double'),
 ('Humidity', 'double'),
 ('Wind_Speed', 'double'),
 ('General_Diffuse_Flows', 'double'),
 ('Diffuse_Flows', 'double'),
 ('Power_Zone_1', 'double'),
 ('Power_Zone_2', 'double'),
 ('Power_Zone_3', 'double'),
 ('Month', 'bigint'),
 ('Hour', 'bigint')]

## Fitting the Model
The first part of this project will be training an elastic net model with out dataset to predict values for Power Zone 3. In an elastic net model, L1 (LASSO) and L2 (Ridge) penalties are combined to improve model predictions and stability. 

Now that we've loaded our dataset and have a good idea of what our data looks like, we can set up the transformations we want to apply to our data before training our model. The first transformation we'll apply is a SQL transformation to cast the Hour variable as a double instead of an integer. Within the same transformation, we will also set the Power_Zone_3 as label. After changing the Hour variable type, we'll apply a binarizer transformation to this variable to distinguish between night and day using 6.5 as the cutoff. Next , we'll use one-hot encoding to encode the Month variable. Additionally, we will run a PCA (Principle Component Analysis) fit on a few of the columns in our dataset. The PCA entails using a VectorAssembler transformation to place the desired variables together in a column followed by using the PCA transformation.

Lastly, we will use the VectorAssembler transformation to combine our desired predictor variables in a features column. 

In [6]:
#SQL transformer to cast Hour variable as DoubleType
sqlTrans = SQLTransformer(
    statement = """
                SELECT *,
                CAST(Hour AS DOUBLE) AS hour_double,
                Power_Zone_3 as label 
                FROM __THIS__
                """)

In [7]:
#Binarize transformer to convert continuous Hour values to binary values 
binarizer = Binarizer(threshold=6.5, inputCol="hour_double", outputCol="hour_binary")

In [8]:
#One-hot encoder to transform Month values to vector values
##StringIndexer transformation to conver Month values 
indexer = StringIndexer(inputCol="Month", outputCol="month_index")
##OneHotEncoder transformation
encoder = OneHotEncoder(inputCols=["Month"], outputCols=["month_vec"])

In [9]:
#PCA transformation 
##VectorAssembler to combine desired columns
pca_assembler = VectorAssembler(inputCols=["Temperature","Humidity","Wind_Speed","General_Diffuse_Flows","Diffuse_Flows"], outputCol="pca_features")
##PCA transformer 
pca = PCA(k=2,inputCol="pca_features", outputCol="pca_results")

In [10]:
#VectorAssembler to put predictors in features column 
assembler_features = VectorAssembler(inputCols=["hour_binary","Power_Zone_1","Power_Zone_2","month_vec","pca_results"], outputCol="features")

Now that we have defined our transformations, we can define the other components of our model. First we'll create an object to define our linear regression model. Then we'll define our parameter grid to set test values of `regParam`, which controls the amount of regularization, and `elasticNetParam`, which defines the balance between L1 and L2 regularization. We will also set up a pipeline with the transformations defined above and our linear regression model. 

In [11]:
#define object for linear regression model 
lr = LinearRegression()
#define parameter grid 
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()
#define transformation pipeline 
pipeline = Pipeline(stages = [sqlTrans, binarizer, indexer, encoder, pca_assembler, pca, assembler_features, lr])

The next step is to set up our `CrossValidator` object and enter in our pipeline, parameter grid, and RMSE regression evaluator. For our cross validation, we'll use 5 folds. Now we can fit our cross validation model.

In [12]:
#set up cross validation 
crossval = CrossValidator(estimator = pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName='rmse'),
                          numFolds=5)

In [13]:
cvModel = crossval.fit(power_df)

26/04/28 17:10:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/28 17:10:07 WARN Instrumentation: [b6cdef7a] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 17:10:09 WARN Instrumentation: [b6cdef7a] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/28 17:10:11 WARN Instrumentation: [148b699c] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 17:10:12 WARN Instrumentation: [148b699c] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/28 17:10:14 WARN Instrumentation: [198ef797] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 17:10:14 WARN Instrumentation: [198ef797] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
2

At this point, we've successfully used cross validation to train an elastic net model! Let's save this model to avoid having to rerun the training every time we reopen our notebook. We can use `.save()` to save our model in a folder, and `.load()` to reload this model when we want to.

In [14]:
cvModel.write().overwrite().save("cvModel")

In [60]:
cvModel = CrossValidatorModel.load("cvModel")

Let's see how our elastic net model performs. After cross validation, the optimal hyperparameters chosen were a `regParam` of 0.05 and an `elasticNetParam` of 0.1. 

In [61]:
#extract last training stage of the best model 
best_model = cvModel.bestModel.stages[-1]
#iterate through paramters in parameter map for the best model
print("Optimal Paramters:")
print("-" * 30)
for param, value in best_model.extractParamMap().items():
    #print regParam and elasticNetParam
    if param.name in [p.name for p in paramGrid[0].keys()]:
        print(f"{param.name}: {value}")

Optimal Paramters:
------------------------------
elasticNetParam: 0.1
regParam: 0.05


We can also take a look at the CV errors for each combination of `regParam` and `elasticNetParam`. We can see that many of the RMSE values eneded up being similar varying only in their decimal point values.

In [39]:
print("CV Errors:")
print("-" * 30)
for params, score in zip(paramGrid, cvModel.avgMetrics):
    param_str = "|".join([f"{param.name}={value}" for param, value in params.items()])
    print(f"{param_str} -- RMSE: {score:.4f}")

CV Errors:
------------------------------
regParam=0.0|elasticNetParam=0.0 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.05 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.1 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.25 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.5 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.75 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.9 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.95 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.98 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.99 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=1.0 -- RMSE: 2147.8759
regParam=0.05|elasticNetParam=0.0 -- RMSE: 2147.8758
regParam=0.05|elasticNetParam=0.05 -- RMSE: 2147.8768
regParam=0.05|elasticNetParam=0.1 -- RMSE: 2147.8751
regParam=0.05|elasticNetParam=0.25 -- RMSE: 2147.8762
regParam=0.05|elasticNetParam=0.5 -- RMSE: 2147.8753
regParam=0.05|elasticNetParam=0.75 -- RMSE: 2147.8756
regParam=0.05|elasticNetParam=0.9 -- RMSE: 2147.8755
regPar

Now we also want to calculate the resulting RMSE when we use our cvModel to predict Power_Zone_3 values of our original dataset. Doing so gives us an RMSE value of 2147.097. 

In [40]:
lr_rmse = RegressionEvaluator().evaluate(cvModel.transform(power_df))
print(f"RMSE: {lr_rmse}")

RMSE: 2147.0973169293934


Our last step with our model will be using it to add a residual column to our dataframe. First, we use our cvModel as a transformation to add a column of predicted values to our dataframe. We also create a column with residuals values showing the deviation of our predicted values from the original values. 

In [15]:
power_df_pred = cvModel.transform(power_df)
power_df_resid = power_df_pred.withColumn("residual",col("label")-col("prediction"))
power_df_resid.select("label","prediction","residual").show(8)

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386|20878.850660788765| -637.886800788765|
|20131.08434|18660.227266544003| 1470.857073455998|
|19668.43373| 18204.75215311452|1463.6815768854794|
|18899.27711|17590.648498339124|1308.6286116608753|
|18442.40964| 16997.30198645687|1445.1076535431312|
|18130.12048| 16517.68672349429|1612.4337565057103|
|17945.06024|16093.246141053492|1851.8140989465064|
|17459.27711|15722.695360253929|1736.5817497460703|
+-----------+------------------+------------------+
only showing top 8 rows


## Streaming Data
In the previous section we trained an elastic net model on our dataset to predict values of Power_Zone_3. With this model, we can now make predictions with new data that we read in from a stream. First we define a schema for the data that will be streamed in, following the schema from our `power_df` dataframe we used in the previous section. Next, we set up our stream using `.readStream` and tell the stream to look for new data in the folder streaming_files. 

In [71]:
#define schema for read in data
myschema = power_df.schema

In [72]:
#read csv files from folder 'streaming_files' following schema 
stream_df = spark.readStream.schema(myschema).format("csv").option("header","true").load("streaming_files")

We also want to set up some tranformations to process our read in data. First, we'll rename the Power_Zone_3 column to label. Then we'll use our previously trained cvModel to predict values for Power_Zone_3 in our new data. At the same time, we'll also calculate the residuals for predicted values and select only the prediction, residual, and label columns from this dataframe. Once we've defined the transformations we want to do on our stream, we can use `.join()` to combine these dataframes and print them together to the console. Then, we'll use `.writeStream()` to write our transformed data stream to the console.

In [73]:
#transformation to rename response variable to label
rename_df = stream_df.withColumnRenamed("Power_Zone_3","label") 

#transformation to predict Power_Zone_3, create residual, and select columns
predict_df = cvModel.transform(stream_df).withColumn("residual",col("label")-col("prediction")).select("prediction", "residual","label")

joined_df=rename_df.join(predict_df, on='label')
writeDF = joined_df.writeStream.outputMode("append").format("console").start()

26/04/29 13:46:47 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-0f30bd37-aaa7-4de6-83fa-d9f913f8a276. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/29 13:46:47 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


At this point we have started our data stream and have definined what transormations we want to make to our data before it is written to the console. In a python file named `produce_stream_data.py`, we have a script that samples 5 records from a larger database and saves them in a csv file for us to read using our stream. Now that we have our stream read and write set up, we can run `produce_stream_data.py` to populate our streaming_files folder with csv files that will be read by the stream.

In [74]:
%run produce_stream_data.py

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|16643.85542|      15.59|   55.76|     0.077|                399.4|        40.61| 28721.01266| 21559.87842|    1|  15|15485.709550000782| 1158.1458699992181|
|28366.64577|      34.94|   29.84|     4.903|                705.0|        203.1| 42180.33296| 29799.78881|    8|  13| 26764.03235718728| 1602.6134128127196|
|16680.72727|      22.18|    37.8|     0.073|                340.6|        320.7| 32476.72766|  20309.5723|    4|

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|          residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+------------------+
|23324.51613|      13.99|    79.8|     0.085|                0.029|        0.145| 37390.97872| 22013.41463|    3|  22|21247.412148139196| 2077.103981860804|
|28177.45455|      17.05|    74.1|     0.068|                0.037|        0.108| 43029.49408| 23081.05906|    4|  21| 26431.70549094781|1745.7490590521884|
|20660.74372|      10.52|   44.61|     0.082|                 0.04|        0.078| 32833.22034| 20057.14286|    2|  23|

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|25733.22176|      26.25|   44.25|     4.908|                469.1|        59.69| 28225.91362| 19078.48101|    7|   9|22187.672712513802|  3545.549047486198|
|14761.83861|      19.78|    88.5|     0.341|                 0.08|        0.115| 27665.84071| 16952.18295|    9|   4|12928.792340733919| 1833.0462692660822|
|28037.81818|      17.17|   53.52|     0.085|                1.745|        1.564| 45038.36383| 24224.84725|    4|

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|10043.69748|       14.1|   41.26|     0.079|                476.9|        39.43| 31196.95817| 25804.23443|   12|  12|11251.185670614257|-1207.4881906142564|
|9968.787515|      16.63|   56.38|     0.077|                229.5|        222.0| 30880.60837| 25947.83676|   12|  16|11093.347970198007|-1124.5604551980068|
|30306.27615|      23.84|   67.86|     4.905|                0.084|          0.1| 36607.57475| 25477.21519|    7|

-------------------------------------------
Batch: 4
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|25342.83401|      24.48|   39.83|     0.082|                 0.38|        0.334| 45186.09836| 26463.15789|    5|  20| 26489.55126589418| -1146.717255894182|
|17413.01205|       19.4|    76.5|     0.081|                0.212|        0.163|     39280.0| 32954.13223|   11|  18|20077.345696509343|-2664.3336465093416|
|25034.30962|      33.67|   35.42|     4.921|                756.0|         95.5| 28876.54485|  31727.8481|    7|

...

-------------------------------------------
Batch: 14
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|14820.67416|      28.19|   41.41|     4.963|                633.8|        65.11| 26996.81416| 15212.05821|    9|  15| 9005.247017286933|  5815.427142713068|
|25002.45226|      14.67|   65.55|     0.084|                0.062|        0.111| 42565.42373| 26669.90881|    2|  20|25203.750929737653|-201.29866973765456|
|33229.84326|      22.28|   50.22|     4.921|                0.091|        0.126| 48375.04994| 31514.25554|    8

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|17297.45455|      14.21|    89.8|     0.061|                44.09|         39.8| 32290.72121| 17831.36456|    4|   9| 19154.26593620477|-1856.8113862047721|
|22624.26778|      24.62|    79.3|     4.926|                0.069|        0.156| 28602.25914| 18326.58228|    7|   4|25289.583577827194|-2665.3157978271956|
|9876.590636|      17.91|   56.54|     0.074|                404.3|         88.5| 32371.10266| 25899.96932|   12

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|15317.41935|      10.52|    84.6|     0.089|                 0.04|        0.137| 23885.61702| 13935.36585|    3|   2|13957.451944284545| 1359.9674057154552|
|26103.57367|      28.32|   38.59|      4.91|                660.1|        123.2| 38824.06215| 26731.99578|    8|  15|24728.632329011478| 1374.9413409885237|
|22245.72012|      24.31|   57.35|      0.28|                22.26|        23.83| 48380.17699| 29791.68399|    9

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|13437.69508|      14.48|    80.7|     0.087|                464.5|         88.4| 33618.25095| 26912.54986|   12|  13|12725.925171370931|  711.7699086290686|
|13221.41538|      21.53|    76.6|     0.068|                127.1|         97.9| 29677.35099| 14676.92308|    6|   8|16633.114813097556| -3411.699433097556|
|21309.04615|      21.92|    75.7|     0.068|                204.3|        163.2| 36772.45033| 21259.45946|    6

-------------------------------------------
Batch: 18
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|24336.72362|      10.01|    82.3|     4.917|                0.099|          0.1| 39526.77966| 23974.46809|    2|  21| 23074.33033752832| 1262.3932824716794|
|16203.63636|      15.69|    87.2|      0.07|                39.22|        33.86| 28874.40258| 16504.27699|    4|   9| 16969.97968247732|  -766.343322477318|
|11681.92771|        6.0|    82.6|     0.087|                35.06|        33.17| 26211.64557| 16730.69909|    1

Once we've read in all of our data, we can stop writing to the console.

In [68]:
writeDF.stop()

26/04/29 13:43:24 WARN DAGScheduler: Failed to cancel job group 2a3202f5-8d87-45d0-8af9-920cc7e88b88. Cannot find active jobs for it.
26/04/29 13:43:24 WARN DAGScheduler: Failed to cancel job group 2a3202f5-8d87-45d0-8af9-920cc7e88b88. Cannot find active jobs for it.
